# Demo Flowers Recognition: RepLKNet (2022) và VGG-16

Notebook này dùng trực tiếp dataset Flowers đã tải về project và các checkpoint fine-tuned trong `results/flowers/`. Hai model nhận cùng một ảnh, cùng preprocessing và cùng mapping 5 lớp hoa.

VGG-16 đại diện cho CNN thuần truyền thống; RepLKNet-31B đại diện cho CNN large-kernel. Demo này phục vụ minh họa trực quan, còn kết luận định lượng lấy từ toàn bộ test split và nhiều seed trong notebook thực nghiệm.

## 1. Khởi tạo môi trường và chọn GPU

Notebook dừng ngay nếu CUDA không khả dụng, tránh vô tình chạy demo bằng CPU.

In [ ]:
import json
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch
from PIL import Image

PROJECT_ROOT = Path.cwd().resolve()
for candidate in [PROJECT_ROOT, *PROJECT_ROOT.parents]:
    if (candidate / 'src' / 'flower_experiment.py').exists():
        PROJECT_ROOT = candidate
        break

sys.path.insert(0, str(PROJECT_ROOT / 'src'))
DATA_DIR = PROJECT_ROOT / 'data' / 'flowers'
RESULTS_DIR = PROJECT_ROOT / 'results' / 'flowers'
DEMO_SEED = 42
DEVICE = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
if DEVICE.type != 'cuda':
    raise RuntimeError('CUDA không khả dụng. Hãy chọn kernel Python (RepLKNet Demo).')

print(f'Project root: {PROJECT_ROOT}')
print(f'Python: {sys.executable}')
print(f'PyTorch: {torch.__version__}')
print(f'Device: {DEVICE}')
print(f'GPU: {torch.cuda.get_device_name(0)}')

## 2. Dataset và các ảnh mẫu

Ảnh được lấy từ test split đã khóa trong `split_manifest.json`, không lấy ngẫu nhiên từ train để tránh minh họa sai chất lượng tổng quát hóa.

In [ ]:
from torchvision.datasets import ImageFolder
from flower_experiment import build_transforms, locate_imagefolder_root

DATA_ROOT = locate_imagefolder_root(DATA_DIR)
_, EVAL_TRANSFORM = build_transforms(image_size=224)
DATASET = ImageFolder(DATA_ROOT, transform=EVAL_TRANSFORM)
MANIFEST_PATH = RESULTS_DIR / 'split_manifest.json'
MANIFEST = json.loads(MANIFEST_PATH.read_text(encoding='utf-8'))
CLASS_NAMES = DATASET.classes
TEST_INDICES = MANIFEST['indices']['test']

selected = {}
for index in TEST_INDICES:
    class_name = CLASS_NAMES[DATASET.targets[index]]
    if class_name not in selected:
        selected[class_name] = DATASET.samples[index][0]

print(f'Dataset root: {DATA_ROOT}')
print(f'Tổng số ảnh: {len(DATASET)}')
print(f'Classes: {CLASS_NAMES}')
print(f'Test split: {len(TEST_INDICES)} ảnh')
print('Ảnh mẫu theo lớp:')
for class_name in CLASS_NAMES:
    print(f'  {class_name}: {selected[class_name]}')

In [ ]:
fig, axes = plt.subplots(1, len(CLASS_NAMES), figsize=(18, 4))
for axis, class_name in zip(axes, CLASS_NAMES):
    image = Image.open(selected[class_name]).convert('RGB')
    axis.imshow(image)
    axis.set_title(class_name)
    axis.axis('off')
fig.suptitle('Một ảnh test đại diện cho mỗi lớp hoa', y=1.02, fontsize=14)
plt.tight_layout()
plt.show()

## 3. Nạp hai checkpoint fine-tuned

Demo sử dụng checkpoint seed 42 đã lưu sau quá trình huấn luyện. Classifier của cả hai model đều có 5 đầu ra tương ứng với dataset Flowers. RepLKNet được structural re-parameterization trước inference.

In [ ]:
import torch.nn as nn
from torchvision.models import vgg16
from replknet_demo import build_replknet, load_checkpoint_flexible, model_stats

VGG_CHECKPOINT = RESULTS_DIR / f'seed_{DEMO_SEED}' / 'vgg16_best.pt'
REPLK_CHECKPOINT = RESULTS_DIR / f'seed_{DEMO_SEED}' / 'replknet31b_best.pt'
if not VGG_CHECKPOINT.exists() or not REPLK_CHECKPOINT.exists():
    raise FileNotFoundError('Chưa có checkpoint Flowers. Hãy chạy Flower_Experiment_Vietnamese.ipynb trước.')

vgg_model = vgg16(weights=None)
vgg_model.classifier[6] = nn.Linear(vgg_model.classifier[6].in_features, len(CLASS_NAMES))
vgg_load = load_checkpoint_flexible(vgg_model, VGG_CHECKPOINT)
vgg_model = vgg_model.cpu().eval()

replk_model, replk_status = build_replknet(
    PROJECT_ROOT,
    model_name='RepLKNet-31B',
    checkpoint_path=None,
    device=torch.device('cpu'),
    num_classes=len(CLASS_NAMES),
    merge_for_inference=False,
)
replk_load = load_checkpoint_flexible(replk_model, REPLK_CHECKPOINT)
replk_model.structural_reparam()
replk_model = replk_model.cpu().eval()

MODELS = {
    'VGG-16 (CNN thuần)': vgg_model,
    'RepLKNet-31B': replk_model,
}
print(f'VGG checkpoint: {VGG_CHECKPOINT.name}')
print(f'RepLKNet checkpoint: {REPLK_CHECKPOINT.name}')
print(f'VGG params: {model_stats(vgg_model)["parameters"]:,}')
print(f'RepLKNet params: {model_stats(replk_model)["parameters"]:,}')

## 4. Cho hai model dự đoán cùng một ảnh

Có thể đổi `EXAMPLE_CLASS` để kiểm tra một lớp khác. Kết quả dưới đây chỉ là minh họa single-image; metrics khoa học vẫn phải đọc từ toàn bộ test split.

In [ ]:
EXAMPLE_CLASS = 'rose' if 'rose' in selected else CLASS_NAMES[0]
EXAMPLE_PATH = Path(selected[EXAMPLE_CLASS])
EXAMPLE_IMAGE = Image.open(EXAMPLE_PATH).convert('RGB')
EXAMPLE_TENSOR = EVAL_TRANSFORM(EXAMPLE_IMAGE).unsqueeze(0)

def predict_flower(model, batch):
    model = model.to(DEVICE).eval()
    with torch.inference_mode():
        probabilities = torch.softmax(model(batch.to(DEVICE)), dim=1)[0].cpu()
    model.cpu()
    torch.cuda.empty_cache()
    top_index = int(probabilities.argmax())
    return top_index, probabilities

predictions = {}
for model_name, model in MODELS.items():
    top_index, probabilities = predict_flower(model, EXAMPLE_TENSOR)
    predictions[model_name] = {
        'predicted_class': CLASS_NAMES[top_index],
        'confidence': float(probabilities[top_index]),
        'probabilities': probabilities.numpy(),
    }

fig, axis = plt.subplots(figsize=(5, 5))
axis.imshow(EXAMPLE_IMAGE)
axis.set_title(f'Ảnh test | nhãn thật: {EXAMPLE_CLASS}')
axis.axis('off')
plt.show()

for model_name, result in predictions.items():
    print(f"{model_name}: {result['predicted_class']} ({result['confidence']:.2%})")

## 4. Vùng đặc trưng mà mỗi model sử dụng

VGG-16 và RepLKNet nhận cùng một tensor ảnh sau preprocessing. Hai ô `CNN thuần nhìn thấy` và `RepLKNet nhìn thấy` vì thế có cùng nội dung pixel; khác biệt được thể hiện bằng Grad-CAM ở hàng dưới, cho biết vùng ảnh đóng góp nhiều nhất vào lớp mà từng model dự đoán.

In [ ]:
import torch.nn as nn

def last_conv_layer(model):
    layer = None
    for module in model.modules():
        if isinstance(module, nn.Conv2d):
            layer = module
    if layer is None:
        raise RuntimeError('Không tìm thấy Conv2d để tính Grad-CAM.')
    return layer

def gradcam(model, batch, target_index):
    model = model.to(DEVICE).eval()
    for module in model.modules():
        if isinstance(module, nn.ReLU):
            module.inplace = False
    target_layer = last_conv_layer(model)
    cache = {}

    def save_activation(_, __, output):
        cache['activation'] = output

    def save_gradient(_, __, grad_output):
        cache['gradient'] = grad_output[0]

    forward_handle = target_layer.register_forward_hook(save_activation)
    backward_handle = target_layer.register_full_backward_hook(save_gradient)
    model.zero_grad(set_to_none=True)
    with torch.enable_grad():
        logits = model(batch.to(DEVICE))
        logits[0, target_index].backward()
    forward_handle.remove()
    backward_handle.remove()

    activation = cache['activation'].detach()
    gradient = cache['gradient'].detach()
    weights = gradient.mean(dim=(2, 3), keepdim=True)
    heatmap = torch.relu((weights * activation).sum(dim=1, keepdim=True))
    heatmap = torch.nn.functional.interpolate(
        heatmap, size=(224, 224), mode='bilinear', align_corners=False
    )[0, 0]
    heatmap = heatmap.cpu()
    heatmap = (heatmap - heatmap.min()) / (heatmap.max() - heatmap.min() + 1e-8)
    model.cpu()
    torch.cuda.empty_cache()
    return heatmap.numpy()

mean = torch.tensor((0.485, 0.456, 0.406)).view(3, 1, 1)
std = torch.tensor((0.229, 0.224, 0.225)).view(3, 1, 1)
input_view = (EXAMPLE_TENSOR[0].cpu() * std + mean).clamp(0, 1).permute(1, 2, 0).numpy()
gradcams = {}
for model_name, model in MODELS.items():
    target_index = CLASS_NAMES.index(predictions[model_name]['predicted_class'])
    gradcams[model_name] = gradcam(model, EXAMPLE_TENSOR, target_index)

fig, axes = plt.subplots(2, 3, figsize=(15, 9))
axes[0, 0].imshow(EXAMPLE_IMAGE)
axes[0, 0].set_title(f'Ảnh gốc | nhãn thật: {EXAMPLE_CLASS}')
axes[0, 1].imshow(input_view)
axes[0, 1].set_title('CNN thuần nhìn thấy\n224×224 + ImageNet normalization')
axes[0, 2].imshow(input_view)
axes[0, 2].set_title('RepLKNet nhìn thấy\n224×224 + ImageNet normalization')
axes[1, 0].text(0.5, 0.5, 'Cùng một tensor\nđầu vào\n\nKhác biệt nằm ở\nvùng đặc trưng được sử dụng', ha='center', va='center', fontsize=13)
axes[1, 0].set_axis_off()
for axis, model_name in zip(axes[1, 1:], MODELS):
    heatmap = gradcams[model_name]
    axis.imshow(input_view)
    axis.imshow(heatmap, cmap='jet', alpha=0.45, vmin=0, vmax=1)
    result = predictions[model_name]
    axis.set_title(f"Grad-CAM: {model_name}\nclass={result['predicted_class']} | confidence={result['confidence']:.2%}")
for axis in axes.ravel():
    axis.axis('off')
fig.suptitle('So sánh ảnh đầu vào và vùng mô hình tập trung', fontsize=16)
plt.tight_layout()
plt.show()

In [ ]:
from IPython.display import HTML, display

probability_rows = []
for model_name, result in predictions.items():
    probability_rows.append('<tr><th>' + model_name + '</th>' + ''.join(
        f'<td>{probability:.2%}</td>' for probability in result['probabilities']
    ) + '</tr>')
probability_html = '<table><caption>Xác suất dự đoán trên cùng một ảnh test</caption>'
probability_html += '<thead><tr><th>Model</th>' + ''.join(f'<th>{name}</th>' for name in CLASS_NAMES) + '</tr></thead>'
probability_html += '<tbody>' + ''.join(probability_rows) + '</tbody></table>'
display(HTML(probability_html))

x = np.arange(len(CLASS_NAMES))
width = 0.35
fig, ax = plt.subplots(figsize=(11, 5))
for offset, (model_name, result) in zip((-width / 2, width / 2), predictions.items()):
    ax.bar(x + offset, result['probabilities'], width, label=model_name)
ax.set_xticks(x, CLASS_NAMES)
ax.set_xlabel('Lớp hoa')
ax.set_ylabel('Xác suất')
ax.set_title(f'Phân bố xác suất | nhãn thật: {EXAMPLE_CLASS}')
ax.legend(title='Model')
plt.tight_layout()
plt.show()

## 5. Metrics tổng hợp của thí nghiệm

Bảng này lấy từ `aggregate_metrics.json`, tức là kết quả trên test split với các seed đã chạy. Đây mới là phần dùng để viết kết luận khoa học; dự đoán một ảnh chỉ mang tính minh họa.

In [ ]:
aggregate = json.loads((RESULTS_DIR / 'aggregate_metrics.json').read_text(encoding='utf-8'))
metric_rows = []
for model_name, values in aggregate.items():
    metric_rows.append(
        '<tr>' + ''.join([
            f'<td>{model_name}</td>',
            f"<td>{values['test_accuracy']['mean']:.4f} ± {values['test_accuracy']['std']:.4f}</td>",
            f"<td>{values['test_macro_f1']['mean']:.4f} ± {values['test_macro_f1']['std']:.4f}</td>",
            f"<td>{values['test_balanced_accuracy']['mean']:.4f} ± {values['test_balanced_accuracy']['std']:.4f}</td>",
            f"<td>{values['latency_mean_ms']['mean']:.3f}</td>",
        ]) + '</tr>'
    )
metrics_html = '<table><thead><tr><th>Model</th><th>Accuracy</th><th>Macro-F1</th><th>Balanced accuracy</th><th>Latency (ms)</th></tr></thead>'
metrics_html += '<tbody>' + ''.join(metric_rows) + '</tbody></table>'
display(HTML(metrics_html))
display(Image.open(RESULTS_DIR / 'comparison_metrics.png'))